# ITS 2122 – Python for Data Science & AI

## Phase 01: Data Ingestion & Clinical Sanitation

### Project: Patient Risk Stratification – Vitality Health Network (VHN)

### Phase Overview

This phase focuses on preparing the raw clinical dataset for reliable
data analysis and patient risk stratification.

The main objectives of this phase are:

1. Load the clinical dataset into a Pandas DataFrame.
2. Perform an initial data quality and structure assessment.
3. Standardize non-standard missing values.
4. Analyze missing data and evaluate the `weight` feature.
5. Apply ID mappings to obtain meaningful descriptions.
6. Exclude encounters associated with expired patients.
7. Remove exact duplicate records while preserving valid patient encounters.

### Expected Outcome

A clean and structured encounter-level dataset that can be
used for the subsequent analytical phases.

In [3]:
# Step 1.1: Import the required libraries

import pandas as pd
import numpy as np

from pathlib import Path

# Display the installed library versions
print("Pandas Version:", pd.__version__)
print("NumPy Version:", np.__version__)

print("\nRequired libraries imported successfully.")

Pandas Version: 3.0.6
NumPy Version: 2.5.4

Required libraries imported successfully.


### 1.2 Loading the Clinical Dataset

The `diabetic_data.csv` file contains patient encounter information
collected from the hospital's electronic health record system.

The dataset is loaded into a Pandas DataFrame for further processing.

The original dataset uses the question mark (`?`) to represent
missing values in certain columns.

The `na_values` parameter converts these non-standard missing-value
markers into standard NumPy `NaN` values during data ingestion.

This ensures consistent missing-value identification throughout
the data cleaning process.

In [9]:
# Step 1.2: Define the dataset location

DATA_DIR = Path("data_files")

DATA_FILE = DATA_DIR / "diabetic_data.csv"

# Verify that the dataset file exists
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Dataset file not found: {DATA_FILE.resolve()}"
    )

# Load the clinical dataset
df_raw = pd.read_csv(
    DATA_FILE,
    na_values=["?"],
    low_memory=False
)

# Display dataset dimensions
print("Clinical dataset loaded successfully!")

print("Number of Rows:", df_raw.shape[0])
print("Number of Columns:", df_raw.shape[1])

# Display the first five records
display(df_raw.head())

Clinical dataset loaded successfully!
Number of Rows: 101766
Number of Columns: 50


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### 1.3 Initial Data Assessment

An initial assessment is performed to understand the structure,
features, data types, and statistical characteristics of the dataset.

The following Pandas methods are used:

1. `shape` – Identifies the number of rows and columns.
2. `columns` – Displays all available feature names.
3. `info()` – Provides information about data types and non-null values.
4. `describe()` – Generates descriptive statistics for dataset features.
5. `head()` – Displays the first five records.

Special attention is given to identifier columns because their
numerical values represent categories rather than measurable quantities.

This assessment helps identify potential data quality issues
before applying further cleaning operations.

In [10]:
# Step 1.3: Perform an initial assessment of the raw dataset

# Display the number of records and features
print("Dataset Dimensions:")
print(df_raw.shape)

# Display all column names
print("\nDataset Column Names:")
print(df_raw.columns.tolist())

# Display data types and non-null counts
print("\nDataset Information:")
df_raw.info()

# Generate descriptive statistics for all columns
print("\nDescriptive Statistical Summary:")
display(df_raw.describe(include="all").T)

# Display the first five records
print("\nFirst Five Records:")
display(df_raw.head())

Dataset Dimensions:
(101766, 50)

Dataset Column Names:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #  

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
encounter_id,101766.0,NaN,NaN,NaN,165201645.622978,102640295.983457,12522.0,84961194.0,152388987.0,230270887.5,443867222.0
patient_nbr,101766.0,NaN,NaN,NaN,54330400.694947,38696359.346534,135.0,23413221.0,45505143.0,87545949.75,189502619.0
race,99493,5,Caucasian,76099,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,101766,3,Female,54708,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,101766,10,[70-80),26068,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weight,3197,9,[75-100),1336,NaN,NaN,NaN,NaN,NaN,NaN,NaN
admission_type_id,101766.0,NaN,NaN,NaN,2.024006,1.445403,1.0,1.0,1.0,3.0,8.0
discharge_disposition_id,101766.0,NaN,NaN,NaN,3.715642,5.280166,1.0,1.0,1.0,4.0,28.0
admission_source_id,101766.0,NaN,NaN,NaN,5.754437,4.064081,1.0,1.0,7.0,7.0,25.0
time_in_hospital,101766.0,NaN,NaN,NaN,4.395987,2.985108,1.0,2.0,4.0,6.0,14.0



First Five Records:


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### 1.4 Loading the ID Mapping Dataset

The `IDs_mapping.csv` file contains descriptions for the
categorical identifier columns in the clinical dataset.

These mappings are required to interpret the numerical
identifiers correctly.

The following identifier columns are considered:

1. `admission_type_id`
2. `discharge_disposition_id`
3. `admission_source_id`

The mapping file is structured into separate sections.
Therefore, each section is extracted and stored as a
separate dictionary for further processing.

This approach improves the interpretability of the clinical
dataset and supports the identification of expired patients.

In [11]:
# Step 1.4: Load and extract ID mappings

import csv

# Define the mapping file location
MAPPING_FILE = DATA_DIR / "IDs_mapping.csv"

# Verify that the mapping file exists
if not MAPPING_FILE.exists():
    raise FileNotFoundError(
        f"Mapping file not found: {MAPPING_FILE.resolve()}"
    )

# Read the mapping file
with MAPPING_FILE.open(
    mode="r",
    encoding="utf-8-sig",
    newline=""
) as file:

    mapping_rows = list(csv.reader(file))

# Define the required mapping sections
required_sections = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

# Store extracted mappings
id_mappings = {}

# Extract each mapping section
for section in required_sections:

    mapping_dict = {}
    section_found = False

    for index, row in enumerate(mapping_rows):

        # Identify the beginning of the required section
        if row and row[0].strip().lower() == section:

            section_found = True

            # Read mapping records until the next section
            for mapping_row in mapping_rows[index + 1:]:

                if not mapping_row or not mapping_row[0].strip():
                    continue

                first_value = mapping_row[0].strip().lower()

                # Stop when another mapping section begins
                if first_value in required_sections:
                    break

                # Store the ID and its description
                if len(mapping_row) >= 2:
                    mapping_id = mapping_row[0].strip()
                    description = mapping_row[1].strip()

                    if mapping_id.isdigit():
                        mapping_dict[mapping_id] = description

            break

    if not section_found:
        raise ValueError(
            f"Mapping section '{section}' was not found."
        )

    id_mappings[section] = mapping_dict

# Display extracted mappings
for section, mapping_dict in id_mappings.items():

    print(f"\n{section}")
    print("-" * 50)

    for mapping_id, description in mapping_dict.items():
        print(f"{mapping_id}: {description}")

print("\nID mappings loaded successfully.")


admission_type_id
--------------------------------------------------
1: Emergency
2: Urgent
3: Elective
4: Newborn
5: Not Available
6: NULL
7: Trauma Center
8: Not Mapped

discharge_disposition_id
--------------------------------------------------
1: Discharged to home
2: Discharged/transferred to another short term hospital
3: Discharged/transferred to a Skilled Nursing Facility (SNF)
4: Discharged/transferred to ICF
5: Discharged/transferred to another type of inpatient care institution
6: Discharged/transferred to home with home health service
7: Left AMA
8: Discharged/transferred to home under care of Home IV provider
9: Admitted as an inpatient to this hospital
10: Neonate discharged to another hospital for neonatal aftercare
11: Expired
12: Still patient or expected to return for outpatient services
13: Hospice / home
14: Hospice / medical facility
15: Discharged/transferred within this institution to Medicare approved swing bed
16: Discharged/transferred/referred another instit

### 1.5 Applying ID Mappings

The numerical identifiers in the clinical dataset represent
categorical information rather than continuous numerical values.

The extracted ID mappings are used to create additional
description columns for better data interpretation.

The following description columns are generated:

- `admission_type_description`
- `discharge_disposition_description`
- `admission_source_description`

The original identifier columns are preserved to maintain
the integrity of the raw clinical data.

These descriptions will also help identify encounters
associated with expired patients.

In [12]:
# Step 1.5: Apply ID mappings to the clinical dataset

# Create a working copy of the original dataset
df_cleaned = df_raw.copy()

# Define the identifier columns and their description columns
mapping_columns = {
    "admission_type_id": "admission_type_description",
    "discharge_disposition_id": "discharge_disposition_description",
    "admission_source_id": "admission_source_description"
}

# Apply the extracted mappings
for id_column, description_column in mapping_columns.items():

    # Convert identifier values into consistent string format
    id_values = pd.to_numeric(
        df_cleaned[id_column],
        errors="coerce"
    ).astype("Int64").astype("string")

    # Add the corresponding descriptions
    df_cleaned[description_column] = id_values.map(
        id_mappings[id_column]
    )

# Display the mapped identifier columns
for id_column, description_column in mapping_columns.items():

    print(f"\n{id_column} and {description_column}")
    display(
        df_cleaned[
            [id_column, description_column]
        ].drop_duplicates().head(15)
    )

print("\nID descriptions applied successfully.")


admission_type_id and admission_type_description


,admission_type_id,admission_type_description
0,6,NULL
1,1,Emergency
5,2,Urgent
6,3,Elective
2043,4,Newborn
3089,5,Not Available
7789,8,Not Mapped
45829,7,Trauma Center



discharge_disposition_id and discharge_disposition_description


,discharge_disposition_id,discharge_disposition_description
0,25,Not Mapped
1,1,Discharged to home
9,3,Discharged/transferred to a Skilled Nursing Fa...
13,6,Discharged/transferred to home with home healt...
29,2,Discharged/transferred to another short term h...
31,5,Discharged/transferred to another type of inpa...
34,11,Expired
82,7,Left AMA
487,10,Neonate discharged to another hospital for neo...
972,4,Discharged/transferred to ICF



admission_source_id and admission_source_description


,admission_source_id,admission_source_description
0,1,Physician Referral
1,7,Emergency Room
5,2,Clinic Referral
8,4,Transfer from a hospital
335,5,Transfer from a Skilled Nursing Facility (SNF)
396,6,Transfer from another health care facility
469,20,Not Mapped
1014,3,HMO Referral
1039,17,NULL
1595,8,Court/Law Enforcement



ID descriptions applied successfully.


### 1.6 Missing Value Analysis

Missing values can affect the accuracy and reliability
of clinical data analysis.

Although the question mark (`?`) was converted into
standard missing values during data ingestion, a detailed
missing value assessment is required.

This section calculates:

1. Total missing values in each column.
2. Percentage of missing values in each column.
3. Columns containing missing data.

The results are arranged in descending order based on
the percentage of missing values.

This analysis helps identify columns that require
special handling before further data processing.

In [13]:
# Step 1.6: Analyze missing values in the dataset

# Calculate the total missing values in each column
missing_counts = df_cleaned.isnull().sum()

# Calculate the percentage of missing values in each column
missing_percentages = (
    df_cleaned.isnull().mean() * 100
)

# Create a missing value summary
missing_summary = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing Percentage (%)": missing_percentages
})

# Sort columns by missing percentage
missing_summary = missing_summary.sort_values(
    by="Missing Percentage (%)",
    ascending=False
)

# Display columns containing missing values
print("Missing Value Analysis:")

display(
    missing_summary[
        missing_summary["Missing Count"] > 0
    ]
)

# Display the total number of missing values
print(
    "\nTotal Missing Values:",
    df_cleaned.isnull().sum().sum()
)

Missing Value Analysis:


,Missing Count,Missing Percentage (%)
weight,98569,96.858479
max_glu_serum,96420,94.746772
A1Cresult,84748,83.277322
medical_specialty,49949,49.082208
payer_code,40256,39.557416
race,2273,2.233555
diag_3,1423,1.398306
diag_2,358,0.351787
diag_1,21,0.020636



Total Missing Values: 374017


### 1.7 Weight Feature Analysis

The `weight` column is examined to determine the extent
of missing information.

A missing value percentage greater than 90% indicates
that the feature contains insufficient information for
reliable analysis.

According to the project requirements, if more than 90%
of the values are missing, the entire `weight` column
must be removed.

This decision is documented as a Data Quality Limitation
because removing the feature results in the loss of
potentially useful clinical information.

If the missing percentage does not exceed 90%, the
column is retained for further consideration.

In [14]:
# Step 1.7: Analyze and handle missing values in the weight column

# Check whether the weight column exists
if "weight" in df_cleaned.columns:

    # Calculate the percentage of missing values
    weight_missing_percentage = (
        df_cleaned["weight"].isnull().mean() * 100
    )

    print("Weight Column Analysis")
    print("-" * 40)

    print(
        "Total Records:",
        len(df_cleaned)
    )

    print(
        "Missing Values:",
        df_cleaned["weight"].isnull().sum()
    )

    print(
        f"Missing Percentage: "
        f"{weight_missing_percentage:.2f}%"
    )

    # Apply the 90% missing value threshold
    if weight_missing_percentage > 90:

        df_cleaned.drop(
            columns=["weight"],
            inplace=True
        )

        print("\nData Quality Limitation:")
        print(
            "The weight column was removed because "
            "more than 90% of its values were missing."
        )

    else:

        print(
            "\nThe weight column is retained because "
            "its missing percentage does not exceed 90%."
        )

else:

    print(
        "The weight column does not exist in the dataset."
    )

Weight Column Analysis
----------------------------------------
Total Records: 101766
Missing Values: 98569
Missing Percentage: 96.86%

Data Quality Limitation:
The weight column was removed because more than 90% of its values were missing.


### 1.8 Handling Deceased Patients

Patient encounters associated with expired patients must be
excluded before performing readmission analysis.

The `discharge_disposition_id` column contains identifiers
representing the patient's discharge destination or status.

The corresponding descriptions from `IDs_mapping.csv` are
used to identify discharge records associated with expired patients.

All discharge disposition descriptions containing the word
"Expired" are identified.

The corresponding patient encounters are removed from the
working dataset.

The original raw dataset is preserved, and the cleaning
operation is performed on `df_cleaned`.

This step helps prevent deceased patients from being included
in subsequent readmission analysis.

In [ ]:
# Step 1.8: Identify and remove encounters of expired patients

# Identify discharge disposition IDs associated with expired patients
expired_disposition_ids = [
    mapping_id
    for mapping_id, description
    in id_mappings["discharge_disposition_id"].items()
    if "expired" in description.lower()
]

# Verify that expired patient mappings were identified
if not expired_disposition_ids:
    raise ValueError(
        "No expired discharge disposition IDs were found "
        "in the ID mapping dataset."
    )

# Display the identified expired discharge disposition IDs
print("Expired Discharge Disposition IDs:")
print(expired_disposition_ids)

# Convert discharge disposition IDs into a consistent format
discharge_ids = (
    pd.to_numeric(
        df_cleaned["discharge_disposition_id"],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)

# Identify encounters associated with expired patients
expired_mask = discharge_ids.isin(expired_disposition_ids)

# Calculate the number of encounters before removal
records_before_removal = len(df_cleaned)

# Count encounters associated with expired patients
expired_records_count = expired_mask.sum()

# Remove expired patient encounters
df_cleaned = df_cleaned.loc[~expired_mask].copy()

# Calculate the number of remaining records
records_after_removal = len(df_cleaned) 

# Display the removal summary
print("\nExpired Patient Removal Summary")
print("-" * 45)

print("Records Before Removal:", records_before_removal)
print("Expired Encounters Removed:", expired_records_count)
print("Records After Removal:", records_after_removal)

# Verify that no expired encounters remain
remaining_expired = (
    pd.to_numeric(
        df_cleaned["discharge_disposition_id"],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
    .isin(expired_disposition_ids)
    .sum()
)

print("\nRemaining Expired Encounters:", remaining_expired)

print("\nExpired patient encounters handled successfully.")

Expired Discharge Disposition IDs:
['11', '19', '20', '21']

Expired Patient Removal Summary
---------------------------------------------
Records Before Removal: 101766
Expired Encounters Removed: 1652
Records After Removal: 100114

Remaining Expired Encounters: 0

Expired patient encounters handled successfully.


### 1.9 Removing Duplicate Records

Duplicate records can affect the accuracy and reliability
of clinical data analysis.

The dataset is examined to identify exact duplicate rows
using all available columns.

Only records that are completely identical are removed.

Multiple encounters belonging to the same patient are valid
clinical records and must be preserved.

Therefore, deduplication is performed using the entire
record rather than the patient identifier (`patient_nbr`).

The number of duplicate records is calculated before removal,
and the dataset is checked again after the cleaning process.

In [16]:
# Step 1.9: Identify and remove exact duplicate records

# Count exact duplicate records before removal
duplicate_count = df_cleaned.duplicated().sum()

# Display the number of duplicate records
print("Duplicate Records Analysis")
print("-" * 40)

print("Records Before Deduplication:", len(df_cleaned))
print("Exact Duplicate Records Found:", duplicate_count)

# Remove exact duplicate records
df_cleaned = df_cleaned.drop_duplicates().copy()

# Display the number of records after deduplication
print("\nDeduplication Summary")
print("-" * 40)

print("Records After Deduplication:", len(df_cleaned))

# Verify that no exact duplicate records remain
remaining_duplicates = df_cleaned.duplicated().sum()

print("Remaining Duplicate Records:", remaining_duplicates)

print("\nExact duplicate records handled successfully.")

Duplicate Records Analysis
----------------------------------------
Records Before Deduplication: 100114
Exact Duplicate Records Found: 0

Deduplication Summary
----------------------------------------
Records After Deduplication: 100114
Remaining Duplicate Records: 0

Exact duplicate records handled successfully.



### 1.10 Converting Categorical Identifier Columns

Some identifier columns contain numerical codes that represent categories rather than measurable quantities.

In this step, the admission type, discharge disposition, and admission source identifier columns are converted into categorical data types.

This improves the representation of categorical information in the clinical dataset.


In [20]:

# Define the identifier columns that represent categorical information
categorical_id_columns = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

# Convert the identifier columns into categorical data types
for column in categorical_id_columns:
    df_cleaned[column] = df_cleaned[column].astype("category")

# Display the updated data types
print("Updated Categorical Identifier Data Types:")
display(df_cleaned[categorical_id_columns].dtypes)


Updated Categorical Identifier Data Types:


admission_type_id           category
discharge_disposition_id    category
admission_source_id         category
dtype: object

### 1.11 Identifying Oral Medication Columns

In this step, the 22 oral medication columns are identified using the dataset column names.

These medication columns are retained separately for future medication efficacy analysis.

In [18]:
# Define the 22 oral medication column names
oral_medication_columns = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "examide",
    "citoglipton",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone"
]

# Check which oral medication columns are available in the dataset
available_medications = [
    column for column in oral_medication_columns
    if column in df_cleaned.columns
]

# Identify medication columns that are missing
missing_medications = [
    column for column in oral_medication_columns
    if column not in df_cleaned.columns
]

print("Number of expected oral medication columns:", len(oral_medication_columns))
print("Number of available oral medication columns:", len(available_medications))

print("\nAvailable Oral Medication Columns:")
print(available_medications)

print("\nMissing Oral Medication Columns:")
print(missing_medications)

# Prepare the oral medication dataset for future analysis
if len(available_medications) == 22:
    df_oral_medications = df_cleaned[available_medications].copy()
    print("\nAll 22 oral medication columns are available.")
    print("Oral medication dataset shape:", df_oral_medications.shape)
else:
    print("\nWarning: The expected 22 oral medication columns were not all found.")

Number of expected oral medication columns: 22
Number of available oral medication columns: 22

Available Oral Medication Columns:
['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']

Missing Oral Medication Columns:
[]

All 22 oral medication columns are available.
Oral medication dataset shape: (100114, 22)


### 1.12 Final Dataset Assessment

In this step, the cleaned clinical dataset is reviewed to verify the results of the data sanitation process.

The final dataset dimensions, missing values, duplicate records, and data types are examined.

In [26]:
# Display the final dataset dimensions
print("Final Dataset Shape:", df_cleaned.shape)

# Display the remaining missing values
print("\nRemaining Missing Values:")
print(df_cleaned.isnull().sum().sum())

# Check for remaining exact duplicate rows
print("\nRemaining Duplicate Rows:", df_cleaned.duplicated().sum())

# Display the final dataset information
print("\nFinal Dataset Information:")
df_cleaned.info()

# Display the first five rows of the cleaned dataset
display(df_cleaned.head())

Final Dataset Shape: (100114, 52)

Remaining Missing Values:
270887

Remaining Duplicate Rows: 0

Final Dataset Information:
<class 'pandas.DataFrame'>
Index: 100114 entries, 0 to 101765
Data columns (total 52 columns):
 #   Column                             Non-Null Count   Dtype   
---  ------                             --------------   -----   
 0   encounter_id                       100114 non-null  category
 1   patient_nbr                        100114 non-null  category
 2   race                               97875 non-null   str     
 3   gender                             100114 non-null  str     
 4   age                                100114 non-null  str     
 5   admission_type_id                  100114 non-null  category
 6   discharge_disposition_id           100114 non-null  category
 7   admission_source_id                100114 non-null  category
 8   time_in_hospital                   100114 non-null  int64   
 9   payer_code                         60523 non-null

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,admission_type_description,discharge_disposition_description,admission_source_description
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,NaN,...,No,No,No,No,No,No,NO,NULL,Not Mapped,Physician Referral
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,NaN,...,No,No,No,No,Ch,Yes,>30,Emergency,Discharged to home,Emergency Room
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,...,No,No,No,No,No,Yes,NO,Emergency,Discharged to home,Emergency Room
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,NaN,...,No,No,No,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,NaN,...,No,No,No,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room


### Step 1: Identify Remaining Missing Values

This step identifies the number and percentage of missing values in each column of the cleaned dataset.

In [21]:
# Calculate the number of missing values in each column
missing_values = df_cleaned.isnull().sum()

# Calculate the percentage of missing values in each column
missing_percentage = (missing_values / len(df_cleaned)) * 100

# Create a summary of missing values
missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage
})

# Display only columns containing missing values
missing_summary = missing_summary[missing_summary["Missing Values"] > 0]

# Sort columns by missing value count in descending order
missing_summary = missing_summary.sort_values(
    by="Missing Values",
    ascending=False
)

# Display the missing values summary
display(missing_summary)

,Missing Values,Missing Percentage
max_glu_serum,94890,94.781949
A1Cresult,83238,83.143217
medical_specialty,49129,49.073057
payer_code,39591,39.545918
race,2239,2.236450
diag_3,1421,1.419382
diag_2,358,0.357592
diag_1,21,0.020976


-------------------------------------------------------------------------------------------------

### Step 1: Convert Identifier Columns to Categorical Data Types

This step converts encounter_id and patient_nbr into categorical data types because they represent unique identifiers rather than continuous numerical measurements.

In [25]:
# Define the identifier columns
identifier_columns = [
    "encounter_id",
    "patient_nbr"
]

# Convert identifier columns into categorical data types
for column in identifier_columns:
    df_cleaned[column] = df_cleaned[column].astype("category")

# Display the updated data types
print("Updated ID Column Data Types:")

display(df_cleaned[id_columns].dtypes)

Updated ID Column Data Types:


encounter_id                category
patient_nbr                 category
admission_type_id           category
discharge_disposition_id    category
admission_source_id         category
dtype: object

### Step 2: Remove All Expired Patients Using ID Mapping

This step uses the discharge disposition mapping file to identify all expired patient records, including IDs 11, 19, 20, and 21. These records are removed to prevent invalid readmission predictions.

In [27]:
# Define all expired patient IDs based on the mapping file
expired_ids = [11, 19, 20, 21]

# Check expired patients in the raw dataset
print("Expired Patients in Raw Dataset:")
print(df_raw["discharge_disposition_id"].isin(expired_ids).sum())

# Check expired patients currently remaining in cleaned dataset
print("\nExpired Patients Before Filtering:")
print(df_cleaned["discharge_disposition_id"].isin(expired_ids).sum())

# Remove all expired patient records
df_cleaned = df_cleaned[
    ~df_cleaned["discharge_disposition_id"].isin(expired_ids)
].copy()

# Reset the dataset index
df_cleaned.reset_index(drop=True, inplace=True)

# Verify expired patient removal
print("\nExpired Patients After Filtering:")
print(df_cleaned["discharge_disposition_id"].isin(expired_ids).sum())

# Display the updated dataset dimensions
print("\nUpdated Dataset Shape:", df_cleaned.shape)

Expired Patients in Raw Dataset:
1652

Expired Patients Before Filtering:
0

Expired Patients After Filtering:
0

Updated Dataset Shape: (100114, 52)


### Step 3: Verify Expired Patient IDs Against the Mapping File

This step verifies the number of records associated with each expired discharge disposition ID in the original dataset.

In [28]:
# Define all expired patient IDs
expired_ids = [11, 19, 20, 21]

# Count records for each expired ID in the raw dataset
print("Expired Patient Counts by ID:")

for expired_id in expired_ids:
    count = (
        df_raw["discharge_disposition_id"] == expired_id
    ).sum()

    print(f"ID {expired_id}: {count}")

# Display the total number of expired patient records
print("\nTotal Expired Patients:")
print(
    df_raw["discharge_disposition_id"]
    .isin(expired_ids)
    .sum()
)

Expired Patient Counts by ID:
ID 11: 1642
ID 19: 8
ID 20: 2
ID 21: 0

Total Expired Patients:
1652


## Phase 01 Conclusion

Phase 01 focused on loading, inspecting, and cleaning the clinical dataset.

The dataset was loaded into a Pandas DataFrame and examined using different data inspection methods. Missing value markers were standardized, and the weight column was evaluated based on its missing value percentage.

Patient encounters related to expired patients were removed using the ID mapping information. Exact duplicate records were also identified and removed while preserving separate encounters belonging to the same patient.

Finally, the 22 oral medication columns were identified and prepared for future medication efficacy analysis.

The cleaned dataset is now ready for the next phase of the project.